In [13]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import h5py
import dbutils
import requests
import os
import pyspark
from pyspark.sql import functions as F
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [14]:
from pyspark.sql import SparkSession
from operator import add

spark_session = SparkSession\
         .builder\
         .master("spark://192.168.2.31:7077") \
         .appName("Music_Profile")\
         .config("spark.dynamicAllocation.enabled", True)\
         .config("spark.dynamicAllocation.shuffleTracking.enabled",True)\
         .config("spark.shuffle.service.enabled", False)\
         .config("spark.dynamicAllocation.executorIdleTimeout","30s")\
         .config("spark.executor.cores",2)\
         .config("spark.driver.port",9999)\
         .config("spark.blockManager.port",10005)\
         .getOrCreate()

# RDD API
spark_context = spark_session.sparkContext
spark_context.setLogLevel("ERROR")

In [15]:
sqlContext = SQLContext(spark_session.sparkContext)


In [16]:
user_data = sqlContext.read.csv("hdfs://192.168.2.31:9000/data/train_triplets.txt", sep="\t", header=False, inferSchema=True).cache()

In [17]:
songs_data = sqlContext.read.csv("hdfs://192.168.2.31:9000/song_metadata_preprocessing.csv/", sep=",", header=False, inferSchema=True).cache()

In [18]:
user_data = user_data.toDF("user_id", "song_id", "play_count")
user_data.show(10)
most_active_user = (
    user_data.groupBy("user_id")
    .agg(F.sum("play_count").alias("Total_plays"))
    .orderBy(F.desc("Total_plays"))
    .limit(1)
)
most_active_user.collect()[0][0]

+--------------------+------------------+----------+
|             user_id|           song_id|play_count|
+--------------------+------------------+----------+
|b80344d063b5ccb32...|SOAKIMP12A8C130995|         1|
|b80344d063b5ccb32...|SOAPDEY12A81C210A9|         1|
|b80344d063b5ccb32...|SOBBMDR12A8C13253B|         2|
|b80344d063b5ccb32...|SOBFNSP12AF72A0E22|         1|
|b80344d063b5ccb32...|SOBFOVM12A58A7D494|         1|
|b80344d063b5ccb32...|SOBNZDC12A6D4FC103|         1|
|b80344d063b5ccb32...|SOBSUJE12A6D4F8CF5|         2|
|b80344d063b5ccb32...|SOBVFZR12A6D4F8AE3|         1|
|b80344d063b5ccb32...|SOBXALG12A8C13C108|         1|
|b80344d063b5ccb32...|SOBXHDL12A81C204C0|         1|
+--------------------+------------------+----------+
only showing top 10 rows



'093cb74eb3c517c5179ae24caf0ebec51b24d2a2'

In [19]:
songs_data = songs_data.toDF("artist","duration", "song_id", "song title", "release")
songs_data.show(10)
songs_data.printSchema()


+--------------------+---------+--------------------+--------------------+-------+
|              artist| duration|             song_id|          song title|release|
+--------------------+---------+--------------------+--------------------+-------+
|           b'Casual'|218.93179|b'SOMZWCG12A8C13C...| b"I Didn't Mean To"|      0|
|     b'The Box Tops'|148.03546|b'SOCIWDW12A8C13D...|        b'Soul Deep'|   1969|
| b'Sonora Santanera'|177.47546|b'SOXVLOJ12AB0189...|  b'Amor De Cabaret'|      0|
|         b'Adam Ant'|233.40363|b'SONHOTT12A8C134...|  b'Something Girls'|   1982|
|              b'Gob'|209.60608|b'SOFSOCN12A8C143...|   b'Face the Ashes'|   2007|
|b'Jeff And Sheri ...| 267.7024|b'SOYMRWW12A6D4FA...|b'The Moon And I ...|      0|
|          b'Rated R'|114.78159|b'SOMJBYD12A6D4F8...|b'Keepin It Real ...|      0|
|b'Tweeterfriendly...|189.57016|b'SOHKNRJ12A6701D...|     b'Drop of Rain'|      0|
| b'Planet P Project'|269.81832|b'SOIAZJW12AB0185...|       b'Pink World'|   1984|
|   

In [20]:
from pyspark.sql import functions as F

def remove_byte_prefix(songs_df):
    for column in songs_df.columns:
        songs_df = songs_df.withColumn(
            column, F.regexp_replace(F.col(column), r"^b[\"']|[\"']$", "")
        )
    return songs_df


user_data = remove_byte_prefix(user_data)
songs_data = remove_byte_prefix(songs_data)

songs_data.show(10)

+--------------------+---------+------------------+--------------------+-------+
|              artist| duration|           song_id|          song title|release|
+--------------------+---------+------------------+--------------------+-------+
|              Casual|218.93179|SOMZWCG12A8C13C480|    I Didn't Mean To|      0|
|        The Box Tops|148.03546|SOCIWDW12A8C13D406|           Soul Deep|   1969|
|    Sonora Santanera|177.47546|SOXVLOJ12AB0189215|     Amor De Cabaret|      0|
|            Adam Ant|233.40363|SONHOTT12A8C13493C|     Something Girls|   1982|
|                 Gob|209.60608|SOFSOCN12A8C143F5D|      Face the Ashes|   2007|
|Jeff And Sheri Ea...| 267.7024|SOYMRWW12A6D4FAB14|The Moon And I (O...|      0|
|             Rated R|114.78159|SOMJBYD12A6D4F8557|Keepin It Real (S...|      0|
|Tweeterfriendly M...|189.57016|SOHKNRJ12A6701D1F8|        Drop of Rain|      0|
|    Planet P Project|269.81832|SOIAZJW12AB01853F1|          Pink World|   1984|
|                 Clp|266.39

In [21]:
merged_df = songs_data.join(user_data, on='song_id', how='inner')
merged_df.show(10)

+------------------+------------------+---------+--------------------+-------+--------------------+----------+
|           song_id|            artist| duration|          song title|release|             user_id|play_count|
+------------------+------------------+---------+--------------------+-------+--------------------+----------+
|SOWEZSI12A81C21CE6|       Gipsy Kings|194.87302|   Tu Quieres Volver|   1987|b80344d063b5ccb32...|         1|
|SODCXXY12AB0187452|        brokeNCYDE| 214.9873|             Freaxxx|   2008|4bd88bfb25263a75b...|         2|
|SOWPAXV12A67ADA046|       Salt-N-Pepa|207.62077|             Push It|   1988|4bd88bfb25263a75b...|        18|
|SOLXDDC12A6701FBFD|            Eminem| 312.2673|            I'm Back|   2000|b64cdd1a0bd907e5e...|         1|
|SONJBQX12A6D4F8382|         Daft Punk|329.53424|             Da Funk|   1995|b64cdd1a0bd907e5e...|         4|
|SONQBUB12A6D4F8ED0|The Rolling Stones|271.49016|Angie (1993 Digit...|      0|b64cdd1a0bd907e5e...|         2|
|

In [22]:
from pyspark.sql import functions as F
def top_songs(merged_data, user, num_songs = 3):
    user_data = merged_data.filter(merged_data['user_id'] == user)
    top_user_songs = user_data.groupBy('song title').agg(F.sum('play_count').alias('Total_plays')).orderBy(F.desc('Total_plays')).limit(num_songs)
    return top_user_songs

def top_artists(merged_data, user, num_artists = 3):
    user_data = merged_data.filter(merged_data['user_id'] == user)
    top_user_artists = user_data.groupBy('artist').agg(F.countDistinct('song_id').alias('number_of_songs')).orderBy(F.desc('number_of_songs')).limit(num_artists)
    return top_user_artists

In [23]:
# Example Usage
top_song = top_songs(merged_df, user = "093cb74eb3c517c5179ae24caf0ebec51b24d2a2")
top_song.show()
top_artist = top_artists(merged_df, user = "093cb74eb3c517c5179ae24caf0ebec51b24d2a2")
top_artist.show()

+--------------------+-----------+
|          song title|Total_plays|
+--------------------+-----------+
|Given Up (Album V...|       30.0|
|     The Small Print|        3.0|
|Never Gonna Give ...|        1.0|
+--------------------+-----------+



[Stage 19:=====================================================>  (22 + 1) / 23]

+-----------+---------------+
|     artist|number_of_songs|
+-----------+---------------+
|Linkin Park|              1|
|Rick Astley|              1|
|   Paramore|              1|
+-----------+---------------+



In [24]:
#
spark_session.stop()